# Problem Description

**Business context and goal.** A renowned German car manufacturer is establishing a ride-hailing platform operated with a fully electrified vehicle fleet, targeting the more receptive US market first. While the client masters vehicle manufacturing, it lacks the tactical and strategic know-how to operate an electric ride-hailing fleet. As the data science team of Bane & Wayne Partners, our goal is to give the client a data-driven understanding of ride-hailing demand for a representative US city in both spatial and temporal resolution, and to translate that understanding into fleet decisions on three levels. The strategic level concerns which districts to enter first. The tactical level concerns how many vehicles to deploy and at what times and locations. The operational level concerns how to charge and reposition the vehicles. Because the fleet is fully electric, operating cost also depends on how each vehicle is charged: charging at the right times keeps recharging cost low while still ensuring enough energy for the working day. This electrification angle connects the project to the course's broader focus on sustainability-oriented analytics, but in this assignment charging is treated narrowly, as a cost-minimisation question for the operator. Since no proprietary ride-hailing data exists yet, we use the City of Chicago taxi trip records (2024 onwards) as a proxy for ride-hailing demand. Taxis are a reasonable stand-in, although they differ from app-based hailing in booking behaviour and market coverage, a limitation we keep in view when drawing conclusions.

**Data-mining goal.** We translate this business goal into a sequence of analytics problems that together form the project. (1) *Data preparation*: turn the large, privacy-restricted raw trip records into clean, spatially and temporally aggregated demand panels, discretising the city with a hexagonal H3 grid and enriching it with external weather data. (2) *Descriptive spatial analytics*: characterise how demand and trip patterns vary across space and time at different resolutions, and locate demand hotspots using Gaussian Mixture Models and kernel density estimation. (3) *Predictive analytics*: frame demand as a supervised regression of the target `demand_count` per spatial unit and time bucket, conditioned on calendar, weather and location. This answers questions such as "what demand can we expect on a rainy Sunday from 10 to 11am at the outskirts of the city", rather than forecasting the next hour from the current one, and it uses Support Vector Machines and feed-forward neural networks. (4) *Prescriptive analytics*: design a smart-charging agent via reinforcement learning that minimises recharging cost while avoiding the risk of running out of energy during the working day. (5) *Synthesis*: consolidate the findings into actionable recommendations for the client, including whether the operator should rely on private or public charging infrastructure. A guiding question throughout is cost-effectiveness. We benchmark each model against simpler baselines and ask whether the added complexity, such as deep learning over a kernel SVM, is worth it for the client. Success is measured with standard regression metrics (MAE, RMSE, R²) relative to a naive benchmark for the predictive models, and by recharging cost versus an energy-shortfall penalty for the charging agent.

# Data description

**Primary data: Chicago taxi trips.** The project's core dataset is the City of Chicago *Taxi Trips (2024–)* public record, used as a proxy for ride-hailing demand. It contains roughly 14.4 million individual trips spanning January 2024 to early April 2026 (extracted May 2026), with one row per trip and 23 raw columns. Each record provides an anonymised vehicle identifier (`Taxi ID`), start and end timestamps rounded to 15-minute intervals, trip duration in seconds (`Trip Seconds`) and distance in miles (`Trip Miles`), the operating company and payment type, and a fare breakdown (`Fare`, `Tips`, `Tolls`, `Extras`, `Trip Total`). Location is recorded at three nested spatial levels: Census Tract, Community Area (Chicago's 77 administrative neighbourhoods), and pickup/dropoff centroid latitude-longitude. For privacy reasons the city suppresses the Census Tract for trips in sparsely travelled areas (missing for about 56% of pickups) and rounds timestamps. This directly shapes which fields are usable as spatial units (see Data preparation). For demand modelling, only the pickup-side time and location fields are required.

**External weather data.** Because weather is a strong short-term driver of mobility, the trip data is enriched with historical weather for Chicago from the Open-Meteo Historical Weather API (lat 41.85, lon -87.65), at hourly and daily resolution over the same 2024-2026 window. The variables include 2 m air temperature in F°, precipitation, rain and snowfall in inches, wind speed in mph, and the WMO weather code, plus daily sunshine duration and precipitation duration in hours. These are aligned to the trip timestamps so that weather-driven demand variation is not misattributed to the calendar.

**Spatial reference.** A GeoJSON of Chicago's 77 community-area boundaries provides the geometry used to map and aggregate trips spatially and to anchor the H3 hexagonal grid introduced in the next section.

# Data preparation

## Data Cleaning

Data is technically cleaned before being analyzed to ensure that all columns have consistent formats and can be processed correctly. The main steps include: (1) standardizing column names; (2) converting timestamps into datetime format; (3) converting numeric and monetary columns into proper numeric types; and (4) checking for duplicate rows. This step ensures we have technically clean, processable data.

## Missing Value Treatment

This section covers two steps: (1) investigating missing values and (2) defining the treatment strategy.

First, we examine the distribution of missing values to assess whether any patterns are present. In this dataset, missing values are not equally distributed across all spatial columns. Census Tract information has a very high share of missing values (55.65%). This is expected because the City of Chicago suppresses Census Tracts for some trips for privacy reasons. Therefore, Census Tracts are not suitable as the main spatial aggregation level. Community Area (2.79%) and centroid coordinates (2.74%) are much more complete and are therefore used as the main spatial units for the model. Crucially, the missing values for these variables are distributed uniformly across months and hours of the day, which means that excluding them does not distort the observed temporal demand pattern.

Dropoff variables have more missing values than pickup variables. Since the demand model focuses on where trips start, missing dropoff values are not used as grounds for removing observations from the main demand dataset.

Missing values are not removed globally, but are handled according to the specific validation or aggregation task. For the main modeling dataset — taxi demand defined by pickup location — we remove observations with null pickup coordinates. For side analyses such as trip duration, we exclude trips where duration equals zero.

## Semantic Validation and Outlier Handling

After analyzing missing values, the next step is to verify whether the available values are meaningful from a business and domain perspective. We assess the reasonableness of trip start and end times, duration, distance, the relationship between distance and price, and trip fares by examining summary statistics (mean, median, and distribution) for each variable. We also confirm that pickup coordinates fall within the geographic boundaries of Chicago.

Based on this inspection, three hard filters are applied:

- **Impossible timestamps** (0.0007%): trips where the start timestamp is later than the end timestamp are excluded as physically impossible, most likely caused by data entry errors or timestamp rounding.
- **Excessive duration** (0.08%): trips with duration above 3 hours are removed, as this exceeds what is operationally plausible for an urban taxi ride in Chicago.
- **Extreme distance** (0.008%): trips with extreme distance values are excluded, as they fall outside any geographically plausible range within the Chicago metro area and are likely caused by GPS errors or data anomalies.

Together, these filters remove less than 0.1% of total trips, ensuring the modeling dataset reflects genuine demand without meaningfully reducing its size.

## Aggregation

The cleaned trip-level data is aggregated along two dimensions: time and space.

**(1) Temporal resolution** — Time buckets are derived from the pickup timestamp. We evaluate four granularities: 1-hour, 4-hour, 8-hour, and daily intervals. Finer resolutions capture more operational detail but produce more volatile demand counts per cell; coarser resolutions are more stable but lose short-term signal. A 4-hour bucket strikes a practical balance — it is granular enough to distinguish morning, midday, evening, and night demand patterns, while remaining stable enough for reliable model training.

**(2) Spatial resolution** — We evaluate two spatial unit types: Community Areas and H3 hexagons. Community Areas are administratively defined and interpretable, but their irregular shapes and varying sizes limit spatial precision. H3 hexagons offer a uniform, standardized grid that can be tuned by resolution level (see Appendix). Resolutions range from R4 (~1,770 km², metro-wide) down to R8 (~0.74 km², a few city blocks). R6 (~36 km², roughly a large neighborhood) is selected as the primary spatial unit: it is fine enough to capture meaningful demand variation across Chicago's districts, while finer resolutions such as R7 or R8 would produce many near-zero cells, making the prediction target sparse and noisy.

**(3) Filling zero-demand combinations** — A standard groupby only produces rows where at least one trip occurred, leaving (location, time) pairs with zero pickups absent from the dataset. These implicit zeros are made explicit via a Cartesian product of all cells and time buckets, so the model can learn from the absence of demand. Trip-characteristic columns are set to NaN for these filled cells.

**Final decision: H3 R6 × 4-hour buckets** is used as the primary resolution for all subsequent modeling. This combination provides neighborhood-level spatial granularity with a meaningful temporal structure, while keeping the dataset dense enough for robust model training.

# Data analytics

## Descriptive Analytics
- Brief recap of demand patterns across spatio-temporal resolutions (hex vs. census tract, hourly vs. 4-hourly)
- GMM/KDE hotspot results (1-2 key maps)

## Predictive Analytics
Analytical methods applied and appropriate performance evaluation (proper choice of measures, benchmarking).

- General info:
    + Validation strategy: how you split train/test (e.g., temporal holdout, not random shuffle, since this is spatio-temporal)
    + Validation metrics: MAE, RMSE, R²

- Support vector machines
    + Model progression. (For reference: linear → kernel (RBF), and how you selected hyperparameters (grid search / manual CV loop — mention why manual, since GridSearchCV + PyTorch deadlock isn't relevant here but the CV strategy is worth stating))
    + Result of the performance evaluation metrics (e.g., RMSE, MAE, R²)
    + Comment on its shortfall

- Neural networks
    + Architecture summary (layers, activation, regularization — dropout/BatchNorm)
    + Training details worth a sentence each: log1p target transform, gradient clipping, hyperparameter tuning approach
    + SHAP feature importance (1 summary plot, 2-3 sentences on top drivers)
    + Performance evaluation metrics (e.g., RMSE, MAE, R²)
    + Spatial error map (H3/GeoPandas) — where does the model struggle geographically (can also added it in the appendix)

- Resolution sensitivity
    + How model's performance varies as we decrease or increase temporal or spatial resolution or when we use census tract

- SVM vs. NN comparison
    + A side-by-side holdout table (SVM vs NN) with MAE, RMSE, R^2 and cost
    + An explicit verdict on whether a deep-learning approach is warranted here
    + Improvement levers for a follow-up project
    + A non-technical translation of the chosen model's error into a fleet-operations implication

Appendix:
- Model result: Report a table across resolutions (hex vs. census tract, different temporal bins)

# Appendix

| Resolution | Avg Area | Rough Scale |
|:---|---:|:---|
| R4 | ~1,770 km² | Large city / metro region |
| R5 | ~253 km² | City district |
| R6 | ~36 km² | Large neighborhood |
| R7 | ~5.2 km² | Neighborhood |
| R8 | ~0.74 km² | A few city blocks |